# Demo 1.1: Setup and Introduction

Set up your environment and verify connectivity to MLflow on Databricks.

In [ ]:
import sys; sys.path.insert(0, "..")

---
## What is MLflow?

Open-source platform for the ML/GenAI lifecycle: tracking, tracing, evaluation, and deployment. Vendor-neutral, OpenTelemetry-compatible, 30+ framework integrations.

---
## Step 1: Verify Installation

In [ ]:
# Verify installation
import mlflow
import openai
print(f"MLflow version: {mlflow.__version__}")
print(f"OpenAI version: {openai.__version__}")

---
## Step 2: Environment Configuration

In [ ]:
# Create a .env file for your API keys (do this once)
# This file should NOT be committed to version control

import os
from pathlib import Path

env_file = Path(".env")

if not env_file.exists():
    print("Creating .env file...")
    with open(env_file, "w") as f:
        f.write("# MLflow GenAI Demo Configuration\n")
        f.write("OPENAI_API_KEY=your-api-key-here\n")
        f.write("\n# MLflow Configuration\n")
        f.write("MLFLOW_TRACKING_URI=databricks\n")
    print("✅ Created .env file. Please edit it and add your OpenAI API or Model Provider keys and URLs.")
else:
    print("✅ .env file already exists.")

In [ ]:
# Install LangChain dependencies
!pip install databricks-langchain langchain-openai -q

In [ ]:
# Load environment variables
from dotenv import load_dotenv

import sys; sys.path.insert(0, "..")
from utils.clnt_utils import is_databricks_ai_gateway_client

load_dotenv()

# Verify if using Databricks as the foundational model provider
use_databricks_ai_gateway = is_databricks_ai_gateway_client()
if use_databricks_ai_gateway:
    print("✅ Using Databricks hosted foundational models via AI Gateway")
else:
    api_key = os.getenv("OPENAI_API_KEY")
    # Verify API key is set (without displaying it)
    if api_key and api_key != "your-api-key-here":
        print("✅ OpenAI API key is configured")
    else:
        print("❌ Please set your OPENAI_API_KEY in the .env file")

---
## Step 3: MLflow UI

The MLflow UI is hosted in your Databricks workspace. No local server needed — just navigate to the workspace URL.

---
## Step 4: MLflow Tracking Setup

In [ ]:
# Set tracking URI for sqlite backend
tracking_uri = "https://dbc-baff2b7f-4402.cloud.databricks.com"
mlflow.set_tracking_uri(tracking_uri)

print(f"MLflow Tracking URI: {mlflow.get_tracking_uri()}")

In [ ]:
# Create a test experiment
experiment_name = "01-setup-verification"
mlflow.set_experiment(experiment_name)

print(f"✅ Created experiment: {experiment_name}")
print(f"   Experiment ID: {mlflow.get_experiment_by_name(experiment_name).experiment_id}")

---
## Step 5: First MLflow Run

In [ ]:
import time

# Start a run
with mlflow.start_run(run_name="setup-test") as run:
    # Example of Log parameters
    mlflow.log_param("test_param", "hello-mlflow")
    mlflow.log_param("framework", "openai")
    
    # Example of Log metrics
    mlflow.log_metric("setup_success", 1.0)
    mlflow.log_metric("timestamp", time.time())
    
    # Example of Log a text artifact
    mlflow.log_text("MLflow GenAI Demo - Setup Complete!", "welcome.txt")
    
    print("✅ Run completed successfully!")
    print(f"   Run ID: {run.info.run_id}")
    print(f"   Experiment ID: {run.info.experiment_id}")

In [ ]:
# Display instructions for starting the UI
print("""
╔══════════════════════════════════════════════════════════╗
║         MLflow UI - Getting Started                     ║
╚══════════════════════════════════════════════════════════╝

To start the MLflow UI, open a NEW terminal and run:

    cd {}
    mlflow ui --port 5000

Then open your browser to:

    https://dbc-baff2b7f-4402.cloud.databricks.com

You should see:
  📊 Your experiments listed on the left
  🔍 Runs with parameters and metrics
  📈 Visualization capabilities

Keep the terminal running while using the UI.
Press Ctrl+C in the terminal to stop the UI server.
""".format(Path.cwd()))

---
## Step 6: Verification

In [ ]:
# Verification script
def verify_setup():
    checks = []
    
    # Check 1: MLflow installed
    try:
        import mlflow
        checks.append(("✅", "MLflow installed", mlflow.__version__))
    except ImportError:
        checks.append(("❌", "MLflow not installed", "N/A"))
    
    # Check 2: OpenAI installed
    try:
        import openai
        checks.append(("✅", "OpenAI SDK installed", openai.__version__))
    except ImportError:
        checks.append(("❌", "OpenAI SDK not installed", "N/A"))
    
    # Check 3: API key configured
    use_databricks = os.getenv("USE_DATABRICKS_CLIENT") == "True"
    if use_databricks:
        print("✅ Using Databricks Workspace and Databricks profile")
        print("✅ Using Databricks hosted foundational models")
        print("✅ Using Databricks workspace client")
    else:
        api_key = os.getenv("OPENAI_API_KEY")
        if api_key and api_key != "your-api-key-here":
            checks.append(("✅", "OpenAI API key configured", "***"))
        else:
            checks.append(("❌", "OpenAI API key not configured", "N/A"))
    
    # Check 4: Tracking URI set
    tracking_uri = mlflow.get_tracking_uri()
    checks.append(("✅", "Tracking URI set", tracking_uri))
    
    # Check 5: Test experiment exists
    try:
        exp = mlflow.get_experiment_by_name("01-setup-verification")
        if exp:
            checks.append(("✅", "Test experiment created", exp.experiment_id))
        else:
            checks.append(("❌", "Test experiment not found", "N/A"))
    except Exception as e:
        checks.append(("❌", "Test experiment not found", str(e)))
    
    
    # Print results
    print("\n" + "="*60)
    print("       SETUP VERIFICATION RESULTS")
    print("="*60 + "\n")
    
    for status, check, detail in checks:
        print(f"{status} {check:.<40} {detail}")
    
    print("\n" + "="*60)
    
    # Overall status
    if all(status == "✅" for status, _, _ in checks):
        print("\n🎉 Setup complete! You're ready to proceed to the next notebook.")
    else:
        print("\n⚠️  Some checks failed. Please review the errors above.")

verify_setup()

In [ ]:
# Quick reference guide
print("""
╔══════════════════════════════════════════════════════════════╗
║           MLflow Quick Reference Guide                       ║
╚══════════════════════════════════════════════════════════════╝

📊 EXPERIMENTS
  mlflow.set_experiment("experiment-name")
  mlflow.get_experiment_by_name("experiment-name")
  mlflow.list_experiments()

🏃 RUNS
  with mlflow.start_run(run_name="my-run"):
      # Your code here
      pass

📝 LOGGING
  mlflow.log_param("key", "value")              # Single value
  mlflow.log_params({"key1": "val1", ...})      # Multiple values
  mlflow.log_metric("accuracy", 0.95)           # Single metric
  mlflow.log_metrics({"acc": 0.95, ...})        # Multiple metrics
  mlflow.log_text(text, "file.txt")             # Text artifact
  mlflow.log_artifact("path/to/file")           # File artifact

🔍 TRACING (GenAI)
  mlflow.openai.autolog()                       # Auto-trace OpenAI
  mlflow.langchain.autolog()                    # Auto-trace LangChain
  @mlflow.trace                                 # Manual tracing decorator

🖥️  UI COMMANDS
  mlflow ui                                     # Start UI (port 5000)
  mlflow ui --port 8080                         # Custom port
  mlflow ui --backend-store-uri                 # Custom backend
  mlflow server --backend-store-uri sqlite:///mlflow.db --port 5000

🔧 CONFIGURATION
  mlflow.set_tracking_uri("uri")                # Set tracking location
  mlflow.get_tracking_uri()                     # Get current URI
  mlflow.set_registry_uri("uri")                # Set model registry

""")